In [ ]:
# If running locally, follow README.md for simple dependency installation.
# If using Google Colab, run this cell and then restart the notebook.
%pip install "xarray[complete]>=2025.1.2" "zarr>=3.0.8" icechunk rioxarray pyproj
# rioxarray and pyproj are used here for working with map projections

In [ ]:
import icechunk
import xarray as xr

storage = icechunk.s3_storage(bucket="dynamical-noaa-hrrr", prefix="noaa-hrrr-analysis/v0.1.0.icechunk/", anonymous=True)
repo = icechunk.Repository.open(storage)
session = repo.readonly_session("main")
ds = xr.open_zarr(session.store, chunks=None)
ds

In [ ]:
# Temperature over time in Denver, CO
ds["temperature_2m"].sel(x=0, y=0, method="nearest").plot()

In [ ]:
# HRRR data is distributed in a projected coordinate reference system and
# we include its details in the dataset metadata (e.g. ds.rio.crs, ds.rio.transform()).

# Use pyproj to transform the longitude and latitude of Chicago, IL into dataset x, y coordinates
import rioxarray  # for .rio accessor
from pyproj import Transformer

lon_lat_to_ds = Transformer.from_crs("EPSG:4326", ds.rio.crs, always_xy=True)
x, y = lon_lat_to_ds.transform(-87.6, 41.9)

ds["temperature_2m"].sel(x=x, y=y, method="nearest").sel(time=slice("2020-06-01", "2020-06-30")).plot()

In [ ]:
# Select a geographic area using (west, south, east, north) longitude/latitude coordinates
(
    ds["temperature_2m"]
    .rio.clip_box(-110, 35, -100, 45, crs="EPSG:4326")
    .sel(time="2019-07-15T18:00")
    .plot(figsize=(6, 8), cmap="Spectral_r")
)

In [ ]:
import pandas as pd

# Select multiple US cities and compare precipitation patterns

cities = pd.DataFrame([
    {"city": "Seattle",     "latitude":  47.6, "longitude": -122.3},
    {"city": "Miami",       "latitude":  25.8, "longitude":  -80.2},
    {"city": "Phoenix",     "latitude":  33.4, "longitude": -112.1},
]).set_index("city")

# Transform coordinates
coords = [lon_lat_to_ds.transform(row.longitude, row.latitude) for _, row in cities.iterrows()]
cities["x"] = [c[0] for c in coords]
cities["y"] = [c[1] for c in coords]
cities_ds = cities[["x", "y"]].to_xarray()

(
  ds["precipitation_surface"]
    .sel(time=slice("2020-01-01", "2020-12-31"))
    .sel(x=cities_ds.x, y=cities_ds.y, method="nearest")
    .resample(time="7d").sum()
    .plot(hue="city", size=6, aspect=3)
)

In [ ]:
from IPython.display import HTML
from matplotlib.animation import FuncAnimation
import matplotlib.pyplot as plt

# An interactive animation of temperature during a summer heat wave

data = (
    ds["temperature_2m"]
    .rio.clip_box(-125, 30, -100, 50, crs="EPSG:4326")
    .sel(time=slice("2021-06-25T00", "2021-07-02T00"))
    .coarsen(x=4, y=4, boundary="trim").mean()
    .load()
)

dpi=150
fig, ax = plt.subplots(figsize=(data.x.size/dpi, data.y.size/dpi), dpi=dpi)
fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
ax.axis("off")

img = ax.imshow(data.isel(time=0), cmap='Spectral_r', vmin=270, vmax=310)
anim = FuncAnimation(fig=fig, frames=data, func=lambda frame: img.set_data(frame), interval=80)

HTML(anim.to_jshtml())